# 🔬 CNN für MNIST — Conv2d, BatchNorm, MaxPool

**Professionelles CNN mit Visualisierung der Trainings-Metriken**

In diesem Notebook lernst du:
- CNN-Architektur mit Conv2d, BatchNorm2d, MaxPool2d, Dropout
- Training mit Train/Test-Metriken pro Epoche
- Visualisierung: Loss-Kurve, Accuracy-Kurve, Beispiel-Vorhersagen
- Reproduzierbarkeit durch feste Seeds

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Reproduzierbarkeit
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

## 1. Konfiguration & Device

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
EPOCHS = 5
LEARNING_RATE = 0.001

print(f"Device: {DEVICE}")

## 2. Datenvorbereitung

MNIST-Bilder werden normalisiert mit dem bekannten Mean/Std des Datensatzes.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Trainingsdaten: {len(train_dataset):,} Bilder")
print(f"Testdaten:      {len(test_dataset):,} Bilder")

## 3. CNN-Architektur

**Architektur:** 2 Conv-Blöcke → Flatten → 2 FC-Layer → Output

Jeder Conv-Block: `Conv2d → BatchNorm2d → ReLU → MaxPool2d`

- **Conv2d**: Lernt räumliche Filter (Kanten, Texturen, Formen)
- **BatchNorm2d**: Stabilisiert Training, erlaubt höhere Lernraten
- **MaxPool2d**: Reduziert räumliche Dimensionen (Downsampling)
- **Dropout**: Verhindert Overfitting

In [ ]:
class MNIST_CNN(nn.Module):
    """CNN für MNIST mit Conv2d, BatchNorm, MaxPool, Dropout."""
    def __init__(self, num_classes=10):
        super().__init__()

        # Block 1: 1×28×28 → 32×14×14
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)

        # Block 2: 32×14×14 → 64×7×7
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)

        # Fully Connected
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Block 1
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        # Block 2
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        # Flatten
        x = x.view(x.size(0), -1)
        # FC
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = MNIST_CNN(num_classes=10).to(DEVICE)
print(model)
print(f"\nParameter: {sum(p.numel() for p in model.parameters()):,}")

## 4. Loss & Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

## 5. Trainings- und Evaluierungsfunktionen

In [ ]:
def train_epoch():
    """Eine Trainings-Epoche. Gibt (Loss, Accuracy%) zurück."""
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total, 100.0 * correct / total


def evaluate():
    """Evaluation auf Testdaten. Gibt Accuracy% zurück."""
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return 100.0 * correct / total

## 6. Training

In [ ]:
train_losses, train_accs, test_accs = [], [], []

print("── Training ──")
for epoch in range(1, EPOCHS + 1):
    loss, train_acc = train_epoch()
    test_acc = evaluate()

    train_losses.append(loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)

    print(f"Epoch {epoch:2d}/{EPOCHS} | Loss: {loss:.4f} | "
          f"Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

print(f"\n✅ Finale Test-Accuracy: {test_accs[-1]:.2f}%")

## 7. Visualisierung

### 7.1 Trainings-Metriken

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(range(1, EPOCHS + 1), train_losses, "b-o", linewidth=2)
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(range(1, EPOCHS + 1), train_accs, "g-o", label="Train", linewidth=2)
axes[1].plot(range(1, EPOCHS + 1), test_accs, "r-s", label="Test", linewidth=2)
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

### 7.2 Beispiel-Vorhersagen

Grün = korrekt, Rot = falsch klassifiziert

In [ ]:
model.eval()
data_iter = iter(test_loader)
images, labels = next(data_iter)
images, labels = images[:10].to(DEVICE), labels[:10]

with torch.no_grad():
    outputs = model(images)
    _, preds = outputs.max(1)

fig2, axes2 = plt.subplots(2, 5, figsize=(12, 5))
axes2 = axes2.flatten()
for i in range(10):
    img = images[i].cpu().squeeze()
    axes2[i].imshow(img, cmap="gray")
    color = "green" if preds[i] == labels[i] else "red"
    axes2[i].set_title(f"Pred: {preds[i].item()} (True: {labels[i].item()})",
                       color=color, fontsize=10)
    axes2[i].axis("off")
fig2.suptitle("Beispiel-Vorhersagen (grün = korrekt, rot = falsch)", fontsize=13)
fig2.tight_layout()
plt.show()

## Zusammenfassung

| Komponente | Zweck |
|-----------|-------|
| **Conv2d** | Lernt räumliche Filter (Kanten, Texturen) |
| **BatchNorm2d** | Stabilisiert & beschleunigt Training |
| **MaxPool2d** | Reduziert Dimensionen, erhöht Receptive Field |
| **Dropout** | Regularisierung gegen Overfitting |
| **CrossEntropyLoss** | Standard-Loss für Klassifikation |
| **Adam** | Adaptiver Optimierer |

**Ergebnis:** ~99% Accuracy auf MNIST mit nur 5 Epochen!